# 📡 State Space Models & the Kalman Filter for Time Series

This notebook introduces **State Space Models (SSM)** and the **Kalman Filter**,
places them in the broader landscape of time-series methods, and compares them
empirically against classical AR/ARMA models and a sequential neural network (LSTM).

### Contents
| # | Topic |
|---|-------|
| 1 | Time-series modelling landscape |
| 2 | Classical models: AR, MA, ARMA, ARIMA |
| 3 | State Space Models — intuition and formulation |
| 4 | The Kalman Filter — prediction, update, and smoothing |
| 5 | Manual Kalman Filter from scratch |
| 6 | Example 1 — Tracking a noisy signal (1-D) |
| 7 | Example 2 — Local Level Model on airline passengers |
| 8 | Example 3 — Structural time series (trend + seasonality) |
| 9 | LSTM baseline — same datasets |
| 10 | Head-to-head comparison: AR / ARMA / Kalman / LSTM |
| 11 | When to use what |

---

## 1. The Time-Series Modelling Landscape

Time-series models can be grouped by the assumptions they make:

```
Time-Series Models
├── Classical statistical
│   ├── AR(p)       — autoregressive
│   ├── MA(q)       — moving average
│   ├── ARMA(p,q)   — combined
│   ├── ARIMA       — with differencing (non-stationary)
│   └── SARIMA      — seasonal ARIMA
│
├── State Space / Probabilistic
│   ├── Local Level Model  (random walk + noise)
│   ├── Local Linear Trend (trend + level + noise)
│   ├── Structural TS      (trend + seasonal + irregular)
│   └── Kalman Filter      (exact inference for linear-Gaussian SSMs)
│
└── Neural / Deep Learning
    ├── RNN
    ├── LSTM / GRU
    └── Temporal Convolutional Network (TCN)
```

| Model family | Interpretable | Uncertainty estimates | Handles missing data | Non-linear |
|--------------|:---:|:---:|:---:|:---:|
| AR / ARMA    | ✅ | ⚠️ (conf. intervals) | ❌ | ❌ |
| State Space (Kalman) | ✅ | ✅ (full covariance) | ✅ | ❌* |
| LSTM / RNN   | ❌ | ❌ (vanilla) | ⚠️ | ✅ |

*Extended/Unscented Kalman Filter handles non-linear systems.

---

## 2. Classical Models: AR, MA, ARMA, ARIMA

### AR(p) — AutoRegressive
$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \varepsilon_t,
\qquad \varepsilon_t \sim \mathcal{N}(0,\sigma^2)$$

### MA(q) — Moving Average
$$y_t = \mu + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q}$$

### ARMA(p,q)
$$y_t = c + \sum_{i=1}^p \phi_i y_{t-i} + \varepsilon_t + \sum_{j=1}^q \theta_j \varepsilon_{t-j}$$

### ARIMA(p,d,q)
Apply differencing $d$ times to make the series stationary, then fit ARMA(p,q).

**Limitations of classical models:**
- Assume stationarity (or require manual differencing)
- No latent (hidden) state — hard to decompose trend / season / noise
- No natural way to propagate uncertainty forward
- Cannot handle missing observations directly

---

## 3. State Space Models (SSM)

A **State Space Model** separates what we *observe* from the hidden *state* that drives it.

### The linear-Gaussian SSM

$$\underbrace{\mathbf{x}_t = F\,\mathbf{x}_{t-1} + B\,\mathbf{u}_t + \mathbf{q}_t}_{\text{state / transition equation}},
\qquad \mathbf{q}_t \sim \mathcal{N}(\mathbf{0}, Q)$$

$$\underbrace{\mathbf{y}_t = H\,\mathbf{x}_t + \mathbf{r}_t}_{\text{observation equation}},
\qquad \mathbf{r}_t \sim \mathcal{N}(\mathbf{0}, R)$$

| Symbol | Meaning | Shape |
|--------|---------|-------|
| $\mathbf{x}_t$ | **latent state** (hidden) | $(m,1)$ |
| $\mathbf{y}_t$ | **observation** (measured) | $(p,1)$ |
| $F$ | state transition matrix | $(m,m)$ |
| $H$ | observation matrix | $(p,m)$ |
| $Q$ | process noise covariance | $(m,m)$ |
| $R$ | measurement noise covariance | $(p,p)$ |

### Why this is powerful
- **Decomposition**: the state vector can encode level, trend, seasonal components simultaneously
- **Uncertainty tracking**: the Kalman filter propagates a full covariance matrix
- **Missing data**: simply skip the update step when $y_t$ is unavailable
- **Any AR/ARMA model can be written as an SSM** (companion form)

---

## 4. The Kalman Filter

The Kalman filter solves **optimal linear filtering**: given noisy observations $y_{1:t}$,
compute the posterior distribution of the state $\mathbf{x}_t$.

Because the model is linear and Gaussian, this posterior is *also* Gaussian:
$$p(\mathbf{x}_t \mid y_{1:t}) = \mathcal{N}(\hat{\mathbf{x}}_{t|t},\; P_{t|t})$$

### Two-step recursion

**Predict** (propagate state forward):
$$\hat{\mathbf{x}}_{t|t-1} = F\,\hat{\mathbf{x}}_{t-1|t-1}$$
$$P_{t|t-1} = F\,P_{t-1|t-1}\,F^\top + Q$$

**Update** (incorporate new observation):
$$\mathbf{v}_t = \mathbf{y}_t - H\,\hat{\mathbf{x}}_{t|t-1}$$
$$S_t = H\,P_{t|t-1}\,H^\top + R$$
$$K_t = P_{t|t-1}\,H^\top\,S_t^{-1}$$
$$\hat{\mathbf{x}}_{t|t} = \hat{\mathbf{x}}_{t|t-1} + K_t\,\mathbf{v}_t$$
$$P_{t|t} = (I - K_t H)\,P_{t|t-1}$$

| Term | Name | Meaning |
|------|------|---------|
| $\mathbf{v}_t$ | innovation | how surprising is the new observation |
| $S_t$ | innovation covariance | expected spread of innovations |
| $K_t$ | Kalman gain | how much to trust the observation vs the prediction |

### Kalman Smoother (RTS)
The **Rauch-Tung-Striebel (RTS)** smoother runs a backward pass after the forward filter,
giving the optimal estimate $\hat{\mathbf{x}}_{t|T}$ using *all* data.

---

## 5. Setup — Install & Import

In [ ]:
!pip install filterpy -q

In [ ]:
# If filterpy is not installed, the cell above installs it.
# You can also run: !pip install filterpy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Classical models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.structural import UnobservedComponents
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Kalman filter
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.titlesize'] = 12
sns.set_theme(style='whitegrid')
np.random.seed(42)
tf.random.set_seed(42)
print('All imports OK')
print(f'TensorFlow: {tf.__version__}')

## 6. Manual Kalman Filter from Scratch

We implement the full predict/update cycle for a **constant-velocity 1-D tracker**
— the classic intro example. The state is $(x, \dot{x})$ (position + velocity).

In [ ]:
class KalmanFilter1D:
    '''
    1-D Kalman filter with constant-velocity state model.
    State: [position, velocity]
    Observation: [position]
    '''
    def __init__(self, dt, process_var, obs_var):
        self.dt = dt
        # State transition matrix
        self.F = np.array([[1, dt],
                           [0,  1]])
        # Observation matrix (we only observe position)
        self.H = np.array([[1, 0]])
        # Process noise covariance
        self.Q = Q_discrete_white_noise(dim=2, dt=dt, var=process_var)
        # Measurement noise covariance
        self.R = np.array([[obs_var]])
        # Initial state and covariance
        self.x = np.zeros((2, 1))
        self.P = np.eye(2) * 500

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x.copy(), self.P.copy()

    def update(self, z):
        z = np.atleast_2d(z).reshape(-1, 1)
        v = z - self.H @ self.x                    # innovation
        S = self.H @ self.P @ self.H.T + self.R    # innovation covariance
        K = self.P @ self.H.T @ np.linalg.inv(S)  # Kalman gain
        self.x = self.x + K @ v
        self.P = (np.eye(len(self.x)) - K @ self.H) @ self.P
        return self.x.copy(), self.P.copy()


# Simulate a noisy position signal
dt    = 1.0
T     = 100
true_pos  = np.cumsum(np.random.randn(T) * 0.5) + 50   # random-walk true position
obs_noise = 3.0
observed  = true_pos + np.random.randn(T) * obs_noise

# Run the filter
kf = KalmanFilter1D(dt=dt, process_var=0.1, obs_var=obs_noise**2)
filtered_pos = []
filtered_std = []

for z in observed:
    kf.predict()
    x, P = kf.update(z)
    filtered_pos.append(x[0, 0])
    filtered_std.append(np.sqrt(P[0, 0]))

filtered_pos = np.array(filtered_pos)
filtered_std = np.array(filtered_std)

# Plot
t = np.arange(T)
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(t, true_pos,    'k-',  lw=1.5, label='True position')
axes[0].plot(t, observed,    'o',   ms=3,  alpha=0.5, color='gray', label='Noisy observations')
axes[0].plot(t, filtered_pos,'r-',  lw=2,  label='Kalman filter estimate')
axes[0].fill_between(t,
                     filtered_pos - 2*filtered_std,
                     filtered_pos + 2*filtered_std,
                     alpha=0.2, color='red', label='95% confidence')
axes[0].set_ylabel('Position')
axes[0].set_title('Manual Kalman Filter — 1-D Position Tracking')
axes[0].legend(fontsize=9)

axes[1].plot(t, filtered_std, color='steelblue', lw=2)
axes[1].set_ylabel('Filter std (uncertainty)')
axes[1].set_xlabel('Time step')
axes[1].set_title('Posterior Standard Deviation — Decreases as Filter Converges')

plt.tight_layout(); plt.show()

mse = np.mean((filtered_pos - true_pos)**2)
mse_raw = np.mean((observed - true_pos)**2)
print(f'MSE raw observations : {mse_raw:.3f}')
print(f'MSE Kalman filtered  : {mse:.3f}')
print(f'SNR improvement      : {mse_raw/mse:.2f}x')

## 7. Example: Airline Passengers — AR vs ARIMA vs State Space

The classic Box-Jenkins airline dataset (monthly passengers 1949-1960) has both
**trend** and **seasonality**. We compare three models:
- ARIMA(2,1,2) — classical differencing approach
- SARIMA(1,1,1)(1,1,1,12) — seasonal ARIMA
- **Structural State Space** (local linear trend + seasonal)

In [ ]:
# Load airline passengers dataset
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
try:
    df = pd.read_csv(url, header=0, index_col=0, parse_dates=True)
    df.columns = ['passengers']
except Exception:
    # Fallback: generate synthetic trend+seasonal data
    t = np.arange(144)
    passengers = (100 + 2*t + 20*np.sin(2*np.pi*t/12) +
                  t/5 * np.sin(2*np.pi*t/12) +
                  np.random.randn(144)*8)
    df = pd.DataFrame({'passengers': passengers.astype(int)})
    df.index = pd.date_range('1949-01', periods=144, freq='MS')

series = df['passengers'].astype(float)
print(f'Series: {len(series)} observations from {series.index[0]} to {series.index[-1]}')

# Train/test split: last 24 months as test
n_test   = 24
train    = series[:-n_test]
test     = series[-n_test:]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(series, color='steelblue', lw=1.5)
axes[0].axvline(train.index[-1], color='crimson', linestyle='--', label='Train/test split')
axes[0].set_title('Airline Passengers (monthly)')
axes[0].set_ylabel('Passengers'); axes[0].legend()

axes[1].plot(np.log(series), color='teal', lw=1.5)
axes[1].set_title('Log-transformed (stabilises variance)')
axes[1].set_ylabel('log(Passengers)')
plt.tight_layout(); plt.show()

In [ ]:
# ACF / PACF to motivate AR order selection
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(np.log(train).diff().dropna(),  lags=30, ax=axes[0], title='ACF  (log, differenced)')
plot_pacf(np.log(train).diff().dropna(), lags=30, ax=axes[1], title='PACF (log, differenced)')
plt.tight_layout(); plt.show()

In [ ]:
log_train = np.log(train)
log_test  = np.log(test)

# ── ARIMA(2,1,2) ─────────────────────────────────────────────────────────
arima_model = ARIMA(log_train, order=(2,1,2)).fit()
arima_fc    = arima_model.forecast(n_test)
arima_fc_orig = np.exp(arima_fc)

# ── SARIMA(1,1,1)(1,1,1,12) ──────────────────────────────────────────────
sarima_model = SARIMAX(log_train,
                       order=(1,1,1),
                       seasonal_order=(1,1,1,12)).fit(disp=False)
sarima_fc    = sarima_model.forecast(n_test)
sarima_fc_orig = np.exp(sarima_fc)

# ── Structural State Space (trend + seasonal) ────────────────────────────
ssm_model = UnobservedComponents(log_train,
                                  level='local linear trend',
                                  seasonal=12).fit(disp=False)
ssm_fc    = ssm_model.forecast(n_test)
ssm_fc_orig = np.exp(ssm_fc)

# ── Plot ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train, color='steelblue', lw=1.5, label='Train')
ax.plot(test,  color='steelblue', lw=1.5, linestyle='--', label='Test (actual)')
ax.plot(test.index, arima_fc_orig,  color='orange',  lw=2, label='ARIMA(2,1,2)')
ax.plot(test.index, sarima_fc_orig, color='green',   lw=2, label='SARIMA(1,1,1)(1,1,1,12)')
ax.plot(test.index, ssm_fc_orig,    color='crimson', lw=2, label='Structural SSM')
ax.axvline(train.index[-1], color='gray', linestyle=':', lw=1)
ax.set_title('Airline Passengers — ARIMA vs SARIMA vs Structural State Space (forecast)')
ax.set_ylabel('Passengers'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# ── Metrics ──────────────────────────────────────────────────────────────
def rmse(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))
def mape(y_true, y_pred): return np.mean(np.abs((y_true - y_pred)/y_true))*100

results = {}
for name, fc in [('ARIMA',   arima_fc_orig),
                 ('SARIMA',  sarima_fc_orig),
                 ('Struct.SSM', ssm_fc_orig)]:
    results[name] = {'RMSE': rmse(test, fc), 'MAPE': mape(test, fc)}
    print(f'{name:15s}  RMSE={results[name]["RMSE"]:8.2f}  MAPE={results[name]["MAPE"]:.2f}%')

## 8. State Space Decomposition — Trend, Seasonal, Irregular

One of the key advantages of structural state space models is transparent **decomposition**
of the series into interpretable components.

In [ ]:
# Fit on full log-series for decomposition
ssm_full = UnobservedComponents(np.log(series),
                                 level='local linear trend',
                                 seasonal=12).fit(disp=False)

# Extract components
res = ssm_full.smoother_results
components = ssm_full.get_prediction()

# Use the summary_frame to get fitted/smoothed values
fitted   = ssm_full.fittedvalues
residuals = np.log(series) - fitted

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(series.index, np.log(series), 'k', lw=1, alpha=0.5, label='Observed (log)')
axes[0].plot(series.index, fitted,          'r', lw=2,            label='SSM fitted')
axes[0].set_title('Observed vs State Space Model Fit')
axes[0].legend(fontsize=9)

axes[1].plot(series.index, fitted, color='steelblue', lw=2)
axes[1].set_title('Smoothed Trend + Seasonal (combined fitted)')
axes[1].set_ylabel('log passengers')

axes[2].plot(series.index, residuals, color='gray', lw=1)
axes[2].axhline(0, color='black', lw=0.8, linestyle='--')
axes[2].set_title('Residuals (Irregular component)')
axes[2].set_ylabel('Residual')

plt.tight_layout(); plt.show()

print('SSM fit summary:')
print(f'  AIC : {ssm_full.aic:.2f}')
print(f'  BIC : {ssm_full.bic:.2f}')
print(f'  RMSE: {rmse(series, np.exp(fitted)):.2f}')

## 9. Kalman Filter Prediction Intervals

A key advantage of the Kalman filter is that it propagates a **full covariance matrix**
through time, giving honest uncertainty estimates that widen into the future.

In [ ]:
# Get prediction with confidence intervals from the structural SSM
pred = ssm_model.get_forecast(n_test)
pred_mean = np.exp(pred.predicted_mean)
pred_ci   = np.exp(pred.conf_int(alpha=0.05))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train, color='steelblue', lw=1.5, label='Train')
ax.plot(test.index,  test,  color='steelblue', lw=1.5, linestyle='--', label='Actual')
ax.plot(pred_mean.index, pred_mean, color='crimson', lw=2, label='SSM forecast (mean)')
ax.fill_between(pred_ci.index,
                pred_ci.iloc[:, 0],
                pred_ci.iloc[:, 1],
                alpha=0.25, color='crimson', label='95% prediction interval')
ax.set_title('Structural SSM — Forecast with Uncertainty Bands')
ax.set_ylabel('Passengers'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# Show how intervals widen with horizon
widths = pred_ci.iloc[:, 1] - pred_ci.iloc[:, 0]
print('Prediction interval width (passengers):')
print(f'  Horizon  1 month  : {widths.iloc[0]:.1f}')
print(f'  Horizon  6 months : {widths.iloc[5]:.1f}')
print(f'  Horizon 12 months : {widths.iloc[11]:.1f}')
print(f'  Horizon 24 months : {widths.iloc[23]:.1f}')

## 10. LSTM Baseline

An **LSTM** (Long Short-Term Memory) is a recurrent neural network that learns
temporal dependencies through gated memory cells.

### LSTM equations (one cell)

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \qquad \text{forget gate}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \qquad \text{input gate}$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c) \qquad \text{candidate cell}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \qquad \text{cell state}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \qquad \text{output gate}$$
$$h_t = o_t \odot \tanh(c_t) \qquad \text{hidden state}$$

The LSTM learns *which* past information to retain, update, or forget.
Unlike the Kalman filter, it has **no explicit noise model** and provides
**no uncertainty estimates** out of the box.

In [ ]:
def make_sequences(data, seq_len):
    '''Create (X, y) sliding-window sequences for supervised learning.'''
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)


# Prepare data — log-scale, min-max scaled
log_series = np.log(series.values).reshape(-1, 1)
scaler     = MinMaxScaler()
log_scaled = scaler.fit_transform(log_series).flatten()

seq_len    = 24   # look back 24 months
n_train    = len(series) - n_test

X_all, y_all = make_sequences(log_scaled, seq_len)
X_tr = X_all[:n_train - seq_len].reshape(-1, seq_len, 1)
y_tr = y_all[:n_train - seq_len]
X_te = X_all[n_train - seq_len:].reshape(-1, seq_len, 1)
y_te = y_all[n_train - seq_len:]

print(f'Train sequences: {X_tr.shape}  |  Test sequences: {X_te.shape}')

# ── Build LSTM ───────────────────────────────────────────────────────────
def build_lstm(seq_len, units=64):
    model = keras.Sequential([
        layers.LSTM(units, return_sequences=True, input_shape=(seq_len, 1)),
        layers.Dropout(0.2),
        layers.LSTM(units // 2),
        layers.Dropout(0.2),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

lstm = build_lstm(seq_len)
lstm.summary()

In [ ]:
history = lstm.fit(
    X_tr, y_tr,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)]
)

# Plot training curve
plt.figure(figsize=(8, 3))
plt.plot(history.history['loss'],     label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('LSTM Training Curve')
plt.legend(); plt.tight_layout(); plt.show()

# Predict and inverse-transform
lstm_pred_scaled = lstm.predict(X_te, verbose=0).flatten()
lstm_pred_log    = scaler.inverse_transform(lstm_pred_scaled.reshape(-1,1)).flatten()
lstm_pred        = np.exp(lstm_pred_log)

lstm_rmse = rmse(test.values, lstm_pred)
lstm_mape = mape(test.values, lstm_pred)
print(f'LSTM  RMSE={lstm_rmse:.2f}  MAPE={lstm_mape:.2f}%')

## 11. Head-to-Head Comparison

We now put all models side by side on the same test window.

In [ ]:
# Collect all forecasts
forecasts = {
    'ARIMA':       arima_fc_orig.values,
    'SARIMA':      sarima_fc_orig.values,
    'Struct. SSM': ssm_fc_orig.values,
    'LSTM':        lstm_pred,
}

# ── Forecast plot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train, color='steelblue', lw=1.5, label='Train')
ax.plot(test.index,  test,  color='black',     lw=2.5, label='Actual', zorder=5)

colors = ['#e67e22', '#27ae60', '#c0392b', '#8e44ad']
for (name, fc), col in zip(forecasts.items(), colors):
    ax.plot(test.index, fc, lw=2, color=col, label=name)

ax.axvline(train.index[-1], color='gray', linestyle=':', lw=1)
ax.set_title('All Models — 24-Month Forecast vs Actual')
ax.set_ylabel('Passengers'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# ── Metrics table ────────────────────────────────────────────────────────
rows = []
for name, fc in forecasts.items():
    rows.append({'Model': name,
                 'RMSE':  round(rmse(test.values, fc), 2),
                 'MAE':   round(mean_absolute_error(test.values, fc), 2),
                 'MAPE':  round(mape(test.values, fc), 2)})

metrics_df = pd.DataFrame(rows).set_index('Model').sort_values('RMSE')
print(metrics_df.to_string())

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics_df['RMSE'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('RMSE (lower = better)'); axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=30)

metrics_df['MAPE'].plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('MAPE % (lower = better)'); axes[1].set_ylabel('MAPE %')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 12. Bonus — Kalman Filter with Missing Data

One of the most practical advantages of the Kalman filter is its **native handling
of missing observations**: simply skip the update step when data is unavailable.
AR/ARMA models and vanilla LSTMs cannot do this without imputation.

In [ ]:
# Synthetic sinusoidal signal with missing chunks
T = 200
t = np.arange(T)
true_signal = np.sin(0.2 * t) + 0.5 * np.sin(0.05 * t)
noisy       = true_signal + np.random.randn(T) * 0.3

# Create missing data: set 20% of observations to NaN randomly + one long gap
observed_missing = noisy.copy()
rand_mask = np.random.rand(T) < 0.15
observed_missing[rand_mask] = np.nan
observed_missing[80:110]    = np.nan   # long gap of 30 steps

# Kalman filter that skips update on NaN
kf2 = KalmanFilter(dim_x=2, dim_z=1)
kf2.F = np.array([[1, 1], [0, 1]])
kf2.H = np.array([[1, 0]])
kf2.Q = Q_discrete_white_noise(dim=2, dt=1, var=0.01)
kf2.R = np.array([[0.09]])
kf2.x = np.array([[0.], [0.]])
kf2.P = np.eye(2) * 1.0

filtered_vals = []
filtered_stds = []

for z in observed_missing:
    kf2.predict()
    if not np.isnan(z):
        kf2.update([[z]])
    filtered_vals.append(kf2.x[0, 0])
    filtered_stds.append(np.sqrt(kf2.P[0, 0]))

filtered_vals = np.array(filtered_vals)
filtered_stds = np.array(filtered_stds)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t, true_signal, 'k-', lw=1.5, label='True signal', zorder=4)
ax.scatter(t, observed_missing, s=12, alpha=0.5, color='gray', label='Observations (NaN shown as gap)')
ax.plot(t, filtered_vals, 'r-', lw=2, label='Kalman estimate', zorder=3)
ax.fill_between(t,
                filtered_vals - 2*filtered_stds,
                filtered_vals + 2*filtered_stds,
                alpha=0.2, color='red', label='95% CI (widens at gap)')
ax.axvspan(80, 110, alpha=0.08, color='orange', label='Missing data gap')
ax.set_title('Kalman Filter — Handles Missing Data Natively (uncertainty widens during gap)')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print('Note: uncertainty band widens during the gap (steps 80-110)')
print(f'  Std at step 79  (last obs before gap) : {filtered_stds[79]:.4f}')
print(f'  Std at step 109 (last step of gap)    : {filtered_stds[109]:.4f}')
print(f'  Std at step 130 (recovery)             : {filtered_stds[130]:.4f}')

## 13. When to Use What

| Criterion | AR/ARMA | ARIMA/SARIMA | State Space (Kalman) | LSTM |
|-----------|:-------:|:------------:|:--------------------:|:----:|
| Short univariate stationary series | ✅ | ✅ | ✅ | ⚠️ |
| Non-stationary (trend/drift) | ❌ | ✅ | ✅ | ✅ |
| Strong seasonality | ❌ | ✅ (SARIMA) | ✅ | ✅ |
| Missing observations | ❌ | ❌ | ✅ | ⚠️ (with tricks) |
| Calibrated uncertainty / intervals | ⚠️ | ⚠️ | ✅ | ❌ |
| Interpretable components | ⚠️ | ⚠️ | ✅ | ❌ |
| Non-linear dynamics | ❌ | ❌ | ⚠️ (EKF/UKF) | ✅ |
| Many features (multivariate) | ⚠️ (VAR) | ⚠️ | ✅ | ✅ |
| Large datasets (>10k samples) | ✅ | ✅ | ✅ | ✅ |
| Small datasets (<100 samples) | ✅ | ✅ | ✅ | ❌ |
| Fast inference | ✅ | ✅ | ✅ | ⚠️ (GPU) |
| Online / streaming updates | ⚠️ | ❌ | ✅ | ❌ |

### Decision guide

```
Is the series stationary?
   No  → Do you need uncertainty intervals?
              Yes → State Space / Kalman filter
              No  → ARIMA/SARIMA (simple) or LSTM (complex non-linear)
   Yes → Is the dataset small (<200 obs)?
              Yes → AR / ARMA / State Space
              No  → Any of the above; LSTM if non-linear patterns suspected

Are there missing observations?
   Yes → Kalman filter (native) or impute first for others

Do you need to explain/decompose the forecast?
   Yes → Structural State Space Model
```

---
*All models above are available in `statsmodels` and `filterpy` — pre-installed on Colab.*
*For production use, consider `pykalman`, `statsmodels.tsa.statespace`, or `numpyro` (probabilistic SSMs).*